In [4]:
import pandas as pd
import os

# DESCARGA DE SECOP II - PROCESOS DE CONTRATACIÓN
dataset_id = "p6dx-8zbt"
sample_rows = 200_000

source_url = f"https://www.datos.gov.co/resource/{dataset_id}.csv?$limit={sample_rows}"
df_secop = pd.read_csv(source_url, low_memory=False)

# 1. Definir la ruta relativa (sube un nivel desde 'notebooks' hacia 'sesion01' y entra a 'data/raw')
directorio_destino = "../data/raw"

# 2. Crear la carpeta si no existe (evita errores al clonar el repo)
os.makedirs(directorio_destino, exist_ok=True)

# 3. Guardar el archivo en la ruta generada
df_secop.to_csv(f"{directorio_destino}/secop_sample.csv", index=False)

print(f"Filas descargadas: {len(df_secop):,}")
print(f"Columnas: {df_secop.shape[1]}")


Filas descargadas: 200,000
Columnas: 59


In [5]:
import pandas as pd
import os

# DESCARGA DE IDEAM - PROCESOS DE CONTRATACIÓN
dataset_id = "57sv-p2fu"
sample_rows = 157_787

source_url = f"https://www.datos.gov.co/resource/{dataset_id}.csv?$limit={sample_rows}"
df_secop = pd.read_csv(source_url, low_memory=False)

# 1. Definir la ruta relativa (sube un nivel desde 'notebooks' hacia 'sesion01' y entra a 'data/raw')
directorio_destino = "../data/raw"

# 2. Crear la carpeta si no existe (evita errores al clonar el repo)
os.makedirs(directorio_destino, exist_ok=True)

# 3. Guardar el archivo en la ruta generada
df_secop.to_csv(f"{directorio_destino}/ideam_sample.csv", index=False)

print(f"Filas descargadas: {len(df_secop):,}")
print(f"Columnas: {df_secop.shape[1]}")

Filas descargadas: 157,787
Columnas: 13


In [10]:
# Generador sintético del caso del acueducto · semilla fija, resultado reproducible
import numpy as np
import pandas as pd
import os  # 1. Importamos os para manejar las rutas

def generate_meter_readings(n_meters=5_000, n_hours=168, seed=42):
    """Genera lecturas horarias de medidores con perfil de consumo realista.

    n_meters: número de puntos de medición
    n_hours:  horas de registro, 168 equivale a una semana
    seed:     semilla fija para reproducibilidad
    """
    rng = np.random.default_rng(seed)
    n_records = n_meters * n_hours

    meter_ids = np.repeat(
        [f"MED-{i:07d}" for i in range(n_meters)], n_hours
    )
    timestamps = np.tile(
        pd.date_range("2024-03-01", periods=n_hours, freq="h"), n_meters
    )

    # Consumo con estacionalidad diaria: picos en la mañana y en la noche
    hour_of_day = np.tile(np.arange(n_hours) % 24, n_meters)
    base_consumption = 0.12 + 0.08 * np.sin((hour_of_day - 6) * np.pi / 12) ** 2
    consumption = np.abs(base_consumption + rng.normal(0, 0.03, n_records))

    quality_flags = rng.choice(
        ["OK", "ESTIMADO", "SIN_SENAL"], size=n_records, p=[0.94, 0.05, 0.01]
    )

    return pd.DataFrame({
        "meter_id": meter_ids,
        "reading_ts": timestamps,
        "consumption_m3": np.round(consumption, 4),
        "quality_flag": quality_flags,
        "municipality": rng.choice(
            ["Girardot", "Espinal", "Flandes", "Ricaurte"], size=n_records
        ),
    })

df_synthetic = generate_meter_readings()

# 2. Definir la ruta relativa hacia la carpeta 'synthetic'
directorio_destino = "../data/synthetic"

# 3. Crear la carpeta si no existe
os.makedirs(directorio_destino, exist_ok=True)

# 4. Guardar el archivo usando la variable del directorio
df_synthetic.to_csv(f"{directorio_destino}/lecturas_acueducto.csv", index=False)

print(f"Registros generados:{len(df_synthetic):,}")

Registros generados:840,000


In [8]:
import os

def get_file_size_gb(file_path):
    """Tamaño real del archivo en disco, en gigabytes."""
    return os.path.getsize(file_path) / (1024 ** 3)

In [19]:
import pandas as pd

def measure_expansion_factor(file_path, **read_options):
    """Mide cuántas veces crece un archivo al cargarse en memoria."""
    size_disk_bytes = os.path.getsize(file_path)
    
    # Manejo del error de codificación (UTF-8 vs Latin-1)
    try:
        df = pd.read_csv(file_path, **read_options)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='latin-1', **read_options)
        
    # Usar deep=True es fundamental para no subestimar el texto
    size_memory_bytes = df.memory_usage(deep=True).sum()
    return size_memory_bytes / size_disk_bytes, df

In [20]:
import psutil

def get_available_memory_gb():
    """Memoria realmente disponible, descontando la que ya está en uso."""
    return psutil.virtual_memory().available / (1024 ** 3)

NIVEL 1 - GUIADO

In [21]:
def profile_source(df, source_name):
    """Perfil mínimo de una fuente cargada."""
    dtype_counts = df.dtypes.value_counts().to_dict()
    object_columns = (df.dtypes == "object").sum()

    return {
        "fuente": source_name,
        "filas": len(df),
        "columnas": df.shape[1],
        "columnas_texto": int(object_columns),
        "proporcion_texto": round(object_columns / df.shape[1], 3),
        "tipos": {str(key): int(value) for key, value in dtype_counts.items()},
    }

In [24]:
import os
import pandas as pd

# Asegúrate de haber ejecutado previamente las funciones de la guía:
# get_file_size_gb, measure_expansion_factor y profile_source

# 1. ACTUALIZAR DICCIONARIO DE FUENTES (con 4 fuentes y rutas relativas)
sources = {
    "SECOP II": "../data/raw/secop_sample.csv",
    "IDEAM": "../data/raw/ideam_sample.csv", # Reemplaza si tu archivo se llama distinto
    "GEIH": "../data/raw/geih/Ocupados.csv", # Reemplaza con el CSV real de la GEIH
    "Sintético Acueducto": "../data/synthetic/lecturas_acueducto.csv" # Nueva fuente agregada
}

measurements = []

for source_name, file_path in sources.items():
    # Si la fuente es GEIH, agregamos sep=';' a las opciones de lectura
    if source_name == "GEIH":
        k_value, df = measure_expansion_factor(file_path, low_memory=False, sep=';')
    else:
        k_value, df = measure_expansion_factor(file_path, low_memory=False)
        
    profile = profile_source(df, source_name)
    profile["tamano_disco_gb"] = round(get_file_size_gb(file_path), 4)
    profile["k"] = round(k_value, 2)
    measurements.append(profile)
    del df # libere memoria antes de cargar la siguiente

# Crear carpeta de resultados si no existe
os.makedirs("../resultados", exist_ok=True)

# Guardar la tabla resumen
df_measurements = pd.DataFrame(measurements)
df_measurements.to_csv("../resultados/mediciones.csv", index=False)

print("--- RESULTADOS DE MEDICIÓN ---")
print(df_measurements[["fuente", "filas", "columnas", "proporcion_texto", "k"]])
print("\n")

--- RESULTADOS DE MEDICIÓN ---
                fuente   filas  columnas  proporcion_texto     k
0             SECOP II  200000        59             0.746  3.09
1                IDEAM  157787        13             0.615  2.85
2                 GEIH   29611       202             0.054  5.37
3  Sintético Acueducto  840000         5             0.800  4.71




In [25]:
def verify_level_1(df_measurements):
    """Verificación automática del nivel 1. Devuelve True si todo pasa."""
    checks = {
        "Hay exactamente 4 fuentes medidas": len(df_measurements) == 4, 
        "Todos los k son mayores que 1": (df_measurements["k"] > 1).all(),
        "Los cuatro k son distintos": df_measurements["k"].nunique() == 4, 
        "No hay tamaños de disco en cero": (df_measurements["tamano_disco_gb"] > 0).all(),
    }
    
    print("--- AUTOVERIFICACIÓN ---")
    for description, passed in checks.items():
        print(f"{'PASA' if passed else 'FALLA'} | {description}")
        
    return all(checks.values())

verify_level_1(df_measurements)

--- AUTOVERIFICACIÓN ---
PASA | Hay exactamente 4 fuentes medidas
PASA | Todos los k son mayores que 1
PASA | Los cuatro k son distintos
PASA | No hay tamaños de disco en cero


True

In [26]:
import math

def compute_threshold_periods(memory_useful_gb, expansion_factor,
                              initial_size_gb, growth_rate):
    """Períodos que faltan para que la fuente exceda la memoria útil.

    Un resultado negativo indica que el umbral ya fue superado.
    """
    if growth_rate <= 0:
        raise ValueError("La tasa de crecimiento debe ser mayor que cero.")

    ratio = memory_useful_gb / (expansion_factor * initial_size_gb)
    return math.log(ratio) / math.log(1 + growth_rate)

In [27]:
import pandas as pd
# Asumimos que la función compute_threshold_periods ya está definida arriba

# 1. Cargar las mediciones que guardaste en el Nivel 1
df_resultados = pd.read_csv("../resultados/mediciones.csv")

# 2. Definir tus supuestos clave (¡Esto es lo que justificas en tu texto!)
memoria_util_8gb = 5.0   # 8GB totales - 3GB del SO
memoria_util_16gb = 13.0 # 16GB totales - 3GB del SO
tasa_crecimiento_g = 0.05 # Supuesto: 5% de crecimiento por período

print("--- HORIZONTE DE SATURACIÓN (t_umbral) ---")
print(f"Supuestos: g = {tasa_crecimiento_g*100}%, Consumo SO = 3GB\n")

for index, row in df_resultados.iterrows():
    fuente = row["fuente"]
    k = row["k"]
    s0 = row["tamano_disco_gb"]
    
    # Calcular umbral para 8GB
    t_8gb = compute_threshold_periods(
        memory_useful_gb=memoria_util_8gb, 
        expansion_factor=k, 
        initial_size_gb=s0, 
        growth_rate=tasa_crecimiento_g
    )
    
    # Calcular umbral para 16GB
    t_16gb = compute_threshold_periods(
        memory_useful_gb=memoria_util_16gb, 
        expansion_factor=k, 
        initial_size_gb=s0, 
        growth_rate=tasa_crecimiento_g
    )
    
    print(f"Fuente: {fuente}")
    print(f"  -> Con 8GB (5GB útiles):  {t_8gb:.1f} períodos")
    print(f"  -> Con 16GB (13GB útiles): {t_16gb:.1f} períodos\n")

--- HORIZONTE DE SATURACIÓN (t_umbral) ---
Supuestos: g = 5.0%, Consumo SO = 3GB

Fuente: SECOP II
  -> Con 8GB (5GB útiles):  43.4 períodos
  -> Con 16GB (13GB útiles): 63.0 períodos

Fuente: IDEAM
  -> Con 8GB (5GB útiles):  80.9 períodos
  -> Con 16GB (13GB útiles): 100.5 períodos

Fuente: GEIH
  -> Con 8GB (5GB útiles):  93.5 períodos
  -> Con 16GB (13GB útiles): 113.1 períodos

Fuente: Sintético Acueducto
  -> Con 8GB (5GB útiles):  66.9 períodos
  -> Con 16GB (13GB útiles): 86.5 períodos



In [28]:
growth_scenarios = [0.01, 0.02, 0.04, 0.08, 0.16]

print("--- ANÁLISIS DE SENSIBILIDAD ---")
for growth_rate in growth_scenarios:
    periods = compute_threshold_periods(
        memory_useful_gb=12,
        expansion_factor=5.4,
        initial_size_gb=7.8,
        growth_rate=growth_rate,
    )
    print(f"g = {growth_rate:.0%} -> t_umbral = {periods:6.1f} períodos")

--- ANÁLISIS DE SENSIBILIDAD ---
g = 1% -> t_umbral = -126.2 períodos
g = 2% -> t_umbral =  -63.4 períodos
g = 4% -> t_umbral =  -32.0 períodos
g = 8% -> t_umbral =  -16.3 períodos
g = 16% -> t_umbral =   -8.5 períodos
